# Danish Agricultural Subsidies - Exploratory Analysis

This notebook explores the silver subsidies datasets to:
1. Profile data quality (nulls, duplicates, formats)
2. Test validation hypotheses (payment × rate vs area)
3. Identify edge cases and anomalies
4. Document findings before building the gold pipeline

## Data Sources
- **støtteoplysninger.naturerhverv.dk** - EU payment data (DKK amounts)
- **Landbrugsstøtte_2023** - Field-level applications (hectares)
- **fvm_grassland_subsidies** - Spatial grassland subsidies
- **fvm_organic_subsidies** - Spatial organic subsidies
- **fvm_environmental_subsidies** - Spatial environmental subsidies

In [ ]:
import io

import numpy as np
import pandas as pd
from google.cloud import storage

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load Data from GCS

In [ ]:
# GCS paths for silver data
GCS_BUCKET = "landbruget-data"
SILVER_PATHS = {
    "stoetteoplysninger": (
        "silver/subsidies/20260110_192737/stoetteoplysninger.naturerhverv.dk_20241223_pii_handled.parquet"
    ),
    "landbrugsstoette": "silver/subsidies/20260110_192737/Landbrugsstoette_2023.parquet",
    "grassland": "silver/fvm_grassland_subsidies_2023/20260110_185622/data.parquet",
    "organic": "silver/fvm_organic_subsidies_2023/20260110_185512/data.parquet",
    "environmental": "silver/fvm_environmental_subsidies_2023/20260110_185742/data.parquet",
}


def load_parquet_from_gcs(bucket_name: str, blob_path: str) -> pd.DataFrame:
    """Load a parquet file from GCS into a pandas DataFrame."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    content = blob.download_as_bytes()
    return pd.read_parquet(io.BytesIO(content))


# Load all datasets
datasets = {}
for name, path in SILVER_PATHS.items():
    datasets[name] = load_parquet_from_gcs(GCS_BUCKET, path)

## 2. Data Profiling

In [ ]:
def profile_dataset(df: pd.DataFrame, name: str):
    """Generate a data profile for a DataFrame."""


    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(1)
    null_df = pd.DataFrame({"nulls": null_counts, "pct": null_pct})

    # Check for CVR column
    cvr_cols = [c for c in df.columns if "cvr" in c.lower() or "vat" in c.lower()]
    if cvr_cols:
        cvr_col = cvr_cols[0]
        # Check format
        df[cvr_col].dropna().astype(str).head(5).tolist()

    return null_df


# Profile all datasets
for name, df in datasets.items():
    profile_dataset(df, name)

## 3. støtteoplysninger Analysis - Double Counting Issue

In [ ]:
stoette = datasets["stoetteoplysninger"].copy()

# Convert amount columns to numeric
amount_cols = [c for c in stoette.columns if "dkk" in c.lower()]
for col in amount_cols:
    stoette[col] = pd.to_numeric(stoette[col], errors="coerce").fillna(0)


In [ ]:
# Flag summary rows
stoette["is_summary_row"] = stoette["measure_type_of_intervention"] == "Total for beneficiary"

# Calculate totals
summary_rows = stoette[stoette["is_summary_row"]]
detail_rows = stoette[~stoette["is_summary_row"]]


# Sum comparison
detail_eagf = detail_rows["amount_by_operation_under_eagf_dkk"].sum()
detail_eafrd = detail_rows["amount_by_operation_under_eafrd_dkk"].sum()
detail_total = detail_eagf + detail_eafrd

summary_total = summary_rows["total_of_the_eu_amount_for_that_beneficiary_dkk"].sum()


# Calculate difference percentage
diff = abs(detail_total - summary_total)
diff_pct = (diff / summary_total) * 100


## 4. Known Subsidy Rates (2023)

Reference rates for validation (from Landbrugsstyrelsen):

In [ ]:
# 2023 Subsidy rates (kr/ha)
RATES_2023 = {
    "grundbetaling": 1999,  # Basic payment
    "biodiversitet_baeredygtighed": 2740,  # Bio-scheme
    "miljo_klimavenligt_graes": 1500,  # Environmental grass
    "varieret_planteproduktion": 615,  # Varied crops
    "ekstensivering_slaet": 3526,  # Extensification with mowing
    # Organic
    "okologisk_basis": 955,
    "okologisk_n_reduktion": 716,
    "okologisk_omlaegning": 1760,
    "okologisk_frugt_baer": 4404,
    # Grassland management (with grundbetaling)
    "pleje_afgraesning_med_grund": 1650,
    "pleje_slaet_med_grund": 850,
    # Grassland management (without grundbetaling)
    "pleje_afgraesning_uden_grund": 2600,
    "pleje_slaet_uden_grund": 1050,
}

for _scheme, _rate in RATES_2023.items():
    pass

## 5. Organic Subsidies Validation

Test hypothesis: `organic_payment ≈ organic_area × 955 kr/ha`

In [ ]:
# Load organic spatial data
organic = datasets["organic"].copy()

In [ ]:
# Normalize CVR
organic["cvr"] = organic["cvr_number"].astype(str).str.zfill(8)

# Aggregate area by CVR
organic_by_cvr = (
    organic.groupby("cvr").agg({"area_ha": "sum", "field_id": "count"}).rename(columns={"field_id": "field_count"})
)


In [ ]:
# Get organic payments from støtteoplysninger
organic_payments = detail_rows[
    detail_rows["measure_type_of_intervention"].str.contains("Organic", case=False, na=False)
].copy()
organic_payments["cvr"] = organic_payments["vat_or_tax_identification_number"].astype(str).str.zfill(8)


# Sum payments by CVR
organic_payments_by_cvr = (
    organic_payments.groupby("cvr")
    .agg({"total_of_the_eu_amount_for_that_beneficiary_dkk": "sum"})
    .rename(columns={"total_of_the_eu_amount_for_that_beneficiary_dkk": "payment_dkk"})
)


In [ ]:
# Join area and payment data
organic_validation = organic_by_cvr.merge(
    organic_payments_by_cvr, left_index=True, right_index=True, how="outer"
).fillna(0)

# Calculate expected payment and variance
organic_validation["expected_payment"] = organic_validation["area_ha"] * RATES_2023["okologisk_basis"]
organic_validation["variance"] = organic_validation["payment_dkk"] - organic_validation["expected_payment"]
organic_validation["variance_pct"] = np.where(
    organic_validation["expected_payment"] > 0,
    organic_validation["variance"] / organic_validation["expected_payment"] * 100,
    np.nan,
)


both_mask = (organic_validation["area_ha"] > 0) & (organic_validation["payment_dkk"] > 0)

## 6. Grassland Subsidies Validation

In [ ]:
# Load grassland spatial data
grassland = datasets["grassland"].copy()
grassland["cvr"] = grassland["cvr_number"].astype(str).str.zfill(8)


In [ ]:
# Check subsidy type codes to understand with/without grundbetaling

In [ ]:
# Aggregate grassland area by CVR
grassland_by_cvr = (
    grassland.groupby("cvr").agg({"area_ha": "sum", "field_id": "count"}).rename(columns={"field_id": "field_count"})
)


# Expected payments (using average of with/without grundbetaling rates)
avg_rate = (RATES_2023["pleje_afgraesning_med_grund"] + RATES_2023["pleje_afgraesning_uden_grund"]) / 2

## 7. Cross-Dataset CVR Overlap Analysis

In [ ]:
# Normalize CVRs across all datasets
stoette_cvrs = set(detail_rows["vat_or_tax_identification_number"].astype(str).str.zfill(8))
organic_cvrs = set(organic["cvr"])
grassland_cvrs = set(grassland["cvr"])




## 8. National Total Validation

Denmark has ~2.6M ha agricultural land. Expected grundbetaling at ~1,999 kr/ha = ~5.2B DKK

In [ ]:
# Check grundbetaling from støtteoplysninger
grundbetaling = detail_rows[
    detail_rows["measure_type_of_intervention"].str.contains("Basic payment", case=False, na=False)
]


# Estimate area from payment
estimated_area = grundbetaling["amount_by_operation_under_eagf_dkk"].sum() / RATES_2023["grundbetaling"]

## 9. Summary & Findings